# DrugReview: the annotation and the selected models against the two human annotators

This notebook reproduces the manuscript's human-verification results on the 500 held-out reviews. It reads the frozen annotation release, the sample key, the two annotators' workbooks, and the audit predictions saved by the rung notebooks; it fits nothing, and the human labels enter no model. Only agreement statistics and counts are printed: the review texts and the annotators' notes are read where a subset needs them but never shown. Results are written to `00_human_validation/outputs/model_agreement/`. The companion notebook `Human_Validation_Agreement_Analysis.ipynb` reports the annotators and the annotation field by field.

| Section | Result | Manuscript |
|---|---|---|
| 1 | The two annotators and the annotation, and each rater's use of the holistic classes | Table 4 Panels A and B |
| 2 | Every selected model against each annotator | Table 6, Tables S7 and S8 |
| 3 | Change in aspect agreement under evidence supervision | Table S15 |
| 4 | Agreement within subsets: uncertainty flags and text cues | Table S5, Section 4.1 |

Agreement is percent agreement, Cohen's kappa, and for the holistic label quadratic-weighted kappa; the 95% intervals come from 1,000 review-level bootstrap resamples with seed 2026, as in the companion notebook.

## 0. Setup: paths and constants

In [1]:
import os, json, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, cohen_kappa_score
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

# ROOT is the folder that holds 00_labelling/, 00_split/, 01_OriginalLabel/, ... (Drive: NLP_Projects/03_DrugReview; local: "01. Colab backup").
if os.path.isdir("/content") and not os.environ.get("DRUGREVIEW_ROOT"):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)

def resolve_project_root():
    override = os.environ.get("DRUGREVIEW_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"DRUGREVIEW_ROOT does not exist: {candidate}")
        return candidate
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")]:
        for candidate in (base, base / "DrugReview_new"):
            if (candidate / "01_OriginalLabel").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set DRUGREVIEW_ROOT to the folder containing 01_OriginalLabel, 00_split, and the run data.")

ROOT = resolve_project_root()
OUT = Path(os.environ.get("DRUGREVIEW_HV_MODEL_OUT", ROOT / "00_human_validation" / "outputs" / "model_agreement")); OUT.mkdir(parents=True, exist_ok=True)

PROTOCOL_ID = "v2_ef356fdf87"
REL = ROOT / "00_labelling" / "output_v2" / PROTOCOL_ID
CLASSES = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE"]
A3 = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
ASPECTS = ["efficacy", "safety", "burden", "cost"]
ENCODERS = ("albert", "biobert")
RUNS = {"03": "03_Full_aspects/outputs/aspect_auxiliary_v2_3class",
        "04": "04_Holistic_all_aspects/outputs/aspect_auxiliary_v2_3class",
        "A0": "ablation/A0_Matched_holistic_control/outputs/aspect_auxiliary_v2_3class",
        "5p": "05_Confirmatory/outputs/multilevel_representation_only_v2",
        "5r": "05_Confirmatory/outputs/multilevel_representation_residual_v2",
        "5j": "05_Confirmatory/outputs/multilevel_probability_bottleneck_v2",
        "5d": "05_Confirmatory/outputs/multilevel_detached_probability_bottleneck_v2",
        "6ff": "06_Semi_confirmatory/outputs/semi_4anchored_4free_v2",
        "6a": "06_Semi_confirmatory/outputs/semi_4anchored_0free_v2",
        "6f": "06_Semi_confirmatory/outputs/semi_0anchored_4free_v2",
        "7e": "07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1",
        "7c": "07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_0"}
NAMES = {"03": "3 aspect baseline", "04": "4 parallel", "A0": "4 no aspect heads", "5p": "5 fusion", "5r": "5 residual",
         "5j": "5 bottleneck, joint", "5d": "5 bottleneck, detached", "6ff": "6 4 anchored + 4 free", "6a": "6 4 anchored + 0 free",
         "6f": "6 0 anchored + 4 free", "7e": "7 evidence loss on", "7c": "7 evidence loss off"}

def test_predictions(run, enc, usecols=None):
    df = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_test_predictions.csv", usecols=usecols)
    return df.sort_values("uniqueID").reset_index(drop=True)

# Narrative-group bootstrap on group-level confusion matrices: each draw resamples the narrative groups of the test partition
# with replacement (numpy default_rng(seed).integers(0, G, G)), so reviews that share a narrative stay together, and every
# model scored on the same draws is paired. 95% percentile intervals.
def confusion_metrics(M):
    """M: (..., K, K) counts, rows = true. Returns weighted F1, macro F1, quadratic-weighted kappa."""
    K = M.shape[-1]; W = np.array([[(i - j) ** 2 for j in range(K)] for i in range(K)], float)
    tp = np.einsum("...ii->...i", M); rows = M.sum(-1); cols = M.sum(-2); n = rows.sum(-1, keepdims=True)
    den = 2 * tp + (cols - tp) + (rows - tp); f1 = np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0.0)
    wf1 = (f1 * rows).sum(-1) / n[..., 0]; mf1 = f1.mean(-1)
    E = rows[..., :, None] * cols[..., None, :] / n[..., None]; qwk = 1 - (W * M).sum((-1, -2)) / (W * E).sum((-1, -2))
    return wf1, mf1, qwk

def group_draws(groups, n, seed):
    codes = pd.factorize(pd.Series(groups))[0]; G = int(codes.max() + 1); rng = np.random.default_rng(seed)
    counts = np.empty((n, G))                              # filled in place: 10,000 x ~19,000 draws stay near 1.5 GB
    for i in range(n):
        counts[i] = np.bincount(rng.integers(0, G, G), minlength=G)
    return codes, G, counts

def boot_metrics(y, preds, groups, K, n=1000, seed=2025):
    """preds: dict name -> predicted ids (aligned with y). Returns dict name -> (point (wf1, mf1, qwk), draws (n, 3))."""
    codes, G, cnt = group_draws(groups, n, seed); y = np.asarray(y); out = {}
    for k, p in preds.items():
        C = np.zeros((G, K * K)); np.add.at(C, (codes, y * K + np.asarray(p)), 1.0)
        out[k] = (np.array(confusion_metrics(C.sum(0).reshape(K, K))), np.stack(confusion_metrics((cnt @ C).reshape(n, K, K)), 1))
    return out

def ci(a, axis=0):
    return np.percentile(a, [2.5, 97.5], axis=axis)

def fmt_ci(x, lo, hi, d=3):
    return f"{x:.{d}f} ({lo:.{d}f}, {hi:.{d}f})"

def save(df, name, **kw):
    df.to_csv(OUT / name, **kw); print(f"written: outputs/{name}")

print("ROOT:", ROOT); print("OUT :", OUT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/NLP_Projects/03_DrugReview
OUT : /content/drive/MyDrive/NLP_Projects/03_DrugReview/00_human_validation/outputs/model_agreement


## 1. The two annotators and the annotation

In [2]:
# The 500 human-verified reviews in the order of the sample key, with the two annotators' labels and the annotation. The annotation
# comes from the frozen release (the key's own label columns predate it and are not used); the key's source_row_id indexes the release.
key = pd.read_csv(ROOT / "00_human_validation" / "subset" / "human_validation_sample_key_500.csv", usecols=["sample_id", "source_row_id"])
rel = pd.read_csv(REL / f"drugscom_plus_ai_labels_{PROTOCOL_ID}.csv", usecols=["uniqueID", "ai_sent5_hard"] + [f"ai_{a}" for a in ASPECTS])
key["uniqueID"] = rel.iloc[key.source_row_id.values]["uniqueID"].values
up = lambda s: s.astype("string").str.strip().str.upper()
H = key[["sample_id", "uniqueID"]].merge(rel, on="uniqueID", validate="one_to_one")
H["LLM"] = up(H.ai_sent5_hard)
for a in ASPECTS: H[f"LLM_{a}"] = up(H[f"ai_{a}"])
for n in (1, 2):
    ann = pd.read_excel(ROOT / "00_human_validation" / "outputs" / f"human_validation_annotator{n}_500.xlsm", sheet_name="Annotation")
    ann["sample_id"] = ann.sample_id.astype(str).str.strip()
    cols = {"human_overall_sentiment": f"A{n}", **{f"human_{a}": f"A{n}_{a}" for a in ASPECTS}, "human_uncertain": f"A{n}_uncertain"}
    H = H.merge(ann[["sample_id"] + list(cols)].rename(columns=cols), on="sample_id", validate="one_to_one")
    for c in cols.values(): H[c] = up(H[c])
assert len(H) == 500 and set(H.A1) <= set(CLASSES) and set(H.A2) <= set(CLASSES)
print(f"{len(H)} reviews; annotation, annotator 1 and annotator 2 labels loaded")

# Agreement as in Human_Validation_Agreement_Analysis.ipynb: percent agreement, Cohen's kappa, and for the holistic label
# quadratic-weighted kappa; 95% intervals for agreement and kappa from 1,000 review-level bootstrap resamples (seed 2026).
def kappa(a, b, labels, weights=None):
    a, b = pd.Series(a).astype("string"), pd.Series(b).astype("string"); m = a.notna() & b.notna() & a.isin(labels) & b.isin(labels)
    a, b = a[m].astype(str).values, b[m].astype(str).values; n = len(a); K = len(labels); ix = {l: i for i, l in enumerate(labels)}
    if n == 0: return np.nan
    C = np.zeros((K, K))
    for x, y in zip(a, b): C[ix[x], ix[y]] += 1
    E = np.outer(C.sum(1), C.sum(0)) / n
    if weights is None:
        po, pe = np.trace(C) / n, np.trace(E) / n; return np.nan if np.isclose(1 - pe, 0) else float((po - pe) / (1 - pe))
    W = (np.abs(np.arange(K)[:, None] - np.arange(K)[None, :]) / (K - 1)) ** (2 if weights == "quadratic" else 1)
    de = (W * E).sum(); return np.nan if np.isclose(de, 0) else float(1 - (W * C).sum() / de)
def compare(left, right, labels, ordinal=False, n_boot=1000, seed=2026):
    l, r = pd.Series(left).astype("string"), pd.Series(right).astype("string"); m = l.notna() & r.notna()
    l, r = l[m].astype(str).reset_index(drop=True), r[m].astype(str).reset_index(drop=True); n = len(l)
    out = {"n": n, "agreement": float((l == r).mean()), "kappa": kappa(l, r, labels), "qwk": kappa(l, r, labels, "quadratic") if ordinal else np.nan}
    rng = np.random.default_rng(seed); idx = np.arange(n); ba, bk = [], []
    for _ in range(n_boot):
        s = rng.choice(idx, size=n, replace=True); ba.append(float((l.iloc[s].values == r.iloc[s].values).mean())); bk.append(kappa(l.iloc[s], r.iloc[s], labels))
    out.update({"agreement_lo": np.nanpercentile(ba, 2.5), "agreement_hi": np.nanpercentile(ba, 97.5), "kappa_lo": np.nanpercentile(bk, 2.5), "kappa_hi": np.nanpercentile(bk, 97.5)})
    return out
def summary(src_hol, src_asp, name, ref=None):
    """Compare a five-class label (src_hol, or None) and aspect labels (src_asp: aspect -> column, or None) with each annotator.
    ref fixes the comparison partner for both columns (used for the annotator-to-annotator row)."""
    row = {"comparator": name}
    for n in (1, 2):
        other = ref if ref else f"A{n}"
        if src_hol is not None:
            c = compare(H[src_hol], H[other], CLASSES, ordinal=True)
            row.update({f"A{n}_agr": round(100 * c["agreement"], 1), f"A{n}_kappa": round(c["kappa"], 3), f"A{n}_qwk": round(c["qwk"], 3),
                        f"A{n}_agr_ci": f"{100 * c['agreement']:.1f} ({100 * c['agreement_lo']:.1f}, {100 * c['agreement_hi']:.1f})",
                        f"A{n}_kappa_ci": f"{c['kappa']:.3f} ({c['kappa_lo']:.3f}, {c['kappa_hi']:.3f})"})
        else:
            row.update({f"A{n}_{k}": "---" for k in ("agr", "kappa", "qwk", "agr_ci", "kappa_ci")})
        for a in ASPECTS:
            row[f"A{n}_{a[0].upper()}"] = round(compare(H[src_asp[a]], H[(ref + "_" + a) if ref else f"A{n}_{a}"], A3, n_boot=0)["kappa"], 3) if src_asp else "---"
    return row
def show(rows, title, ci=False):
    t = pd.DataFrame(rows)
    cols = ["comparator"] + ([f"A{n}_{k}" for n in (1, 2) for k in ("agr_ci", "kappa_ci")] if ci else [f"A{n}_{k}" for n in (1, 2) for k in ("agr", "kappa", "qwk", "E", "S", "B", "C")])
    print(f"\n{title}\n" + t[cols].to_string(index=False)); return t

panelA = [summary("A1", {a: f"A1_{a}" for a in ASPECTS}, "Other annotator", ref="A2"), summary("LLM", {a: f"LLM_{a}" for a in ASPECTS}, "LLM annotation")]
tA = show(panelA, "Table 4, Panel A: the two annotators and the annotation (the first row compares annotator 1 with annotator 2 in both columns)")
show(panelA, "Table 4, Panel A, with 95% intervals", ci=True); save(tA, "panelA_annotators_annotation.csv", index=False)

500 reviews; annotation, annotator 1 and annotator 2 labels loaded


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, ou


Table 4, Panel A: the two annotators and the annotation (the first row compares annotator 1 with annotator 2 in both columns)
     comparator  A1_agr  A1_kappa  A1_qwk  A1_E  A1_S  A1_B  A1_C  A2_agr  A2_kappa  A2_qwk  A2_E  A2_S  A2_B  A2_C
Other annotator    44.4     0.320   0.770 0.509 0.619 0.368 0.704    44.4     0.320   0.770 0.509 0.619 0.368 0.704
 LLM annotation    43.4     0.312   0.792 0.379 0.587 0.251 0.720    75.0     0.673   0.887 0.495 0.606 0.398 0.806

Table 4, Panel A, with 95% intervals
     comparator         A1_agr_ci          A1_kappa_ci         A2_agr_ci          A2_kappa_ci
Other annotator 44.4 (40.2, 48.4) 0.320 (0.272, 0.366) 44.4 (40.2, 48.4) 0.320 (0.272, 0.366)
 LLM annotation 43.4 (39.0, 47.4) 0.312 (0.264, 0.358) 75.0 (71.0, 78.2) 0.673 (0.622, 0.715)
written: outputs/panelA_annotators_annotation.csv


In [3]:
# Holistic class use of each rater on the 500 reviews (Table 4, Panel B): the share of the reviews assigned to each class.
use = pd.DataFrame({"Annotator 1": H.A1.value_counts(normalize=True), "Annotator 2": H.A2.value_counts(normalize=True),
                    "LLM annotation": H.LLM.value_counts(normalize=True)}).reindex(CLASSES).fillna(0).T.mul(100).round(1)
print("holistic class use (% of the 500 reviews)\n", use.to_string()); save(use, "class_use.csv", index_label="rater")

holistic class use (% of the 500 reviews)
                 VERY_NEGATIVE  NEGATIVE  NEUTRAL  POSITIVE  VERY_POSITIVE
Annotator 1              12.6      18.0     28.0      28.8           12.6
Annotator 2              23.6      20.2      5.4      22.8           28.0
LLM annotation           26.6      17.0      4.0      22.0           30.4
written: outputs/class_use.csv


## 2. The selected models against each annotator

Rung 1 predicts the binned rating and rung 3 has no holistic output; the configurations without aspect heads have no aspect columns.

In [4]:
# The selected models on the same 500 reviews, against each annotator separately. Rungs 1 and 2 cover the seven model families;
# rungs 3-7 the two encoders. A model without a holistic output (rung 3) or without aspect heads is marked ---.
FAM = {"lr": "LR", "rf": "RF", "lgbm": "LightGBM", "gru": "GRU", "cnn": "CNN", "albert": "ALBERT", "biobert": "BioBERT"}
def attach(df, cols):
    d = df[df.uniqueID.isin(H.uniqueID)].drop_duplicates("uniqueID").set_index("uniqueID")
    for src, dst in cols.items(): H[dst] = up(H.uniqueID.map(d[src]))
rows = []
for rung, folder in ((1, "01_OriginalLabel"), (2, "02_4ominiLabel")):
    ml = pd.read_csv(ROOT / folder / "outputs" / "ml_models_predictions_audit.csv"); dl = pd.read_csv(ROOT / folder / "outputs" / "dl_models_predictions_audit.csv")
    attach(ml, {f"{m}_pred": f"r{rung}_{m}" for m in ("lr", "rf", "lgbm")}); attach(dl, {f"{m}_pred": f"r{rung}_{m}" for m in ("gru", "cnn", "albert", "biobert")})
    for m, name in FAM.items(): rows.append({"encoder": m, **summary(f"r{rung}_{m}", None, f"Rung {rung} {name}")})
LABELS = {"03": "Rung 3", "04": "Rung 4", "A0": "no aspect heads", "5p": "Rung 5 fusion", "5r": "residual", "5j": "bottleneck, joint", "5d": "bottleneck, detached",
          "6ff": "Rung 6, 4+4", "6a": "4+0", "6f": "0+4", "7e": "Rung 7 evidence (eta=1)", "7c": "control (eta=0)"}
for run, label in LABELS.items():
    for enc in ENCODERS:
        ap = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_audit_predictions.csv"); tag = f"{run}_{enc}"
        has_hol = "overall_pred" in ap; has_asp = all(f"{a}_pred" in ap for a in ASPECTS)
        attach(ap, {**({"overall_pred": tag} if has_hol else {}), **({f"{a}_pred": f"{tag}_{a}" for a in ASPECTS} if has_asp else {})})
        name = label + (f" {'ALBERT' if enc == 'albert' else 'BioBERT'}" if run in ("03", "04") else "")
        rows.append({"encoder": enc, **summary(tag if has_hol else None, {a: f"{tag}_{a}" for a in ASPECTS} if has_asp else None, name)})
allm = pd.DataFrame(rows)
for enc, title in (("albert", "Table 6: selected ALBERT models"), ("biobert", "Table S7: selected BioBERT models")):
    sel = allm[(allm.encoder == enc) & ~allm.comparator.str.match(r"Rung [12] (LR|RF|LightGBM|GRU|CNN)$")]
    show(sel.to_dict("records"), title)
show(allm.to_dict("records"), "Table S8: every selected model, holistic label, with 95% intervals", ci=True)
save(allm, "models_vs_annotators.csv", index=False)

/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1649: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, ou


Table 6: selected ALBERT models
             comparator A1_agr A1_kappa A1_qwk   A1_E   A1_S   A1_B   A1_C A2_agr A2_kappa A2_qwk   A2_E   A2_S   A2_B   A2_C
          Rung 1 ALBERT   38.6    0.261  0.722    ---    ---    ---    ---   58.8    0.463  0.812    ---    ---    ---    ---
          Rung 2 ALBERT   44.2    0.318  0.791    ---    ---    ---    ---   68.4    0.588  0.858    ---    ---    ---    ---
          Rung 3 ALBERT    ---      ---    ---  0.372  0.589  0.256  0.753    ---      ---    ---  0.488  0.627  0.425  0.815
          Rung 4 ALBERT   44.0    0.319  0.777  0.345  0.658    0.2  0.745   67.6    0.578  0.857  0.486  0.629  0.383  0.797
        no aspect heads   42.6    0.304   0.79    ---    ---    ---    ---   67.8    0.578  0.852    ---    ---    ---    ---
          Rung 5 fusion   45.4    0.334  0.789  0.371  0.671  0.204  0.742   66.6    0.565  0.842  0.454  0.665  0.399  0.813
               residual   43.8    0.314   0.78  0.327   0.65  0.218  0.749   68.0    

## 3. Evidence supervision and agreement with the annotators

Cohen's kappa of the rung-7 aspect heads with the evidence loss on minus off, paired on the same review draws.

In [5]:
# Change in agreement with each annotator under evidence supervision: Cohen's kappa of the rung-7 aspect heads with the evidence
# loss on minus off, paired review-level bootstrap, 1,000 draws (seed 2026), on the 500 reviews.
ann={n:pd.read_excel(ROOT/f"00_human_validation/outputs/human_validation_annotator{n}_500.xlsm",sheet_name="Annotation") for n in (1,2)}
hmap={"NEGATIVE":0,"NEUTRAL":1,"POSITIVE":2}; rng=np.random.default_rng(2026); draws=[rng.integers(0,500,500) for _ in range(1000)]; rows=[]
for enc in ("albert","biobert"):
    p={0:pd.read_csv(ROOT/RUNS["7c"]/enc/f"{enc}_audit_predictions.csv"),1:pd.read_csv(ROOT/RUNS["7e"]/enc/f"{enc}_audit_predictions.csv")}
    for n in (1,2):
        a=ann[n][["sample_id"]+[f"human_{x}" for x in ASPECTS]].merge(key[["sample_id","uniqueID"]],on="sample_id")
        for asp in ASPECTS:
            d=a[["uniqueID",f"human_{asp}"]].merge(p[0][["uniqueID",f"{asp}_pred_id"]],on="uniqueID").merge(p[1][["uniqueID",f"{asp}_pred_id"]],on="uniqueID",suffixes=("_0","_1")).dropna(subset=[f"human_{asp}"])
            y=d[f"human_{asp}"].astype(str).str.upper().map(hmap).values; y0=d[f"{asp}_pred_id_0"].values; y1=d[f"{asp}_pred_id_1"].values
            k=lambda yy,pp: cohen_kappa_score(yy,pp,labels=[0,1,2]); k0,k1=k(y,y0),k(y,y1)
            boot=[k(y[i],y1[i])-k(y[i],y0[i]) for i in (idx[idx<len(d)] for idx in draws)]; lo,hi=np.percentile(boot,[2.5,97.5])
            rows.append({"encoder":enc,"annotator":n,"aspect":asp,"n":len(d),"kappa_eta0":round(k0,3),"kappa_eta1":round(k1,3),"delta":round(k1-k0,3),"ci_lo":round(lo,3),"ci_hi":round(hi,3)})
s7 = pd.DataFrame(rows); print(s7.to_string(index=False)); save(s7, "evidence_supervision_kappa_change.csv", index=False)

encoder  annotator   aspect   n  kappa_eta0  kappa_eta1  delta  ci_lo  ci_hi
 albert          1 efficacy 500       0.327       0.365  0.038 -0.011  0.081
 albert          1   safety 500       0.631       0.621 -0.010 -0.055  0.032
 albert          1   burden 500       0.233       0.226 -0.007 -0.038  0.024
 albert          1     cost 500       0.746       0.772  0.025 -0.013  0.063
 albert          2 efficacy 500       0.436       0.451  0.015 -0.030  0.062
 albert          2   safety 500       0.645       0.628 -0.017 -0.064  0.027
 albert          2   burden 500       0.398       0.411  0.012 -0.033  0.065
 albert          2     cost 500       0.809       0.824  0.014 -0.021  0.049
biobert          1 efficacy 500       0.374       0.364 -0.010 -0.052  0.029
biobert          1   safety 500       0.593       0.585 -0.008 -0.047  0.028
biobert          1   burden 500       0.243       0.220 -0.024 -0.060  0.013
biobert          1     cost 500       0.723       0.709 -0.014 -0.052  0.024

## 4. Agreement within subsets

In [6]:
# Agreement within subsets of the 500 reviews: by each annotator's uncertainty flag, and by two cues in the review text where the
# human and annotation rubrics could part (withdrawal; change over time). Percent agreement and quadratic-weighted kappa on the
# holistic label, Cohen's kappa on each aspect. Only counts and agreement statistics are shown; the annotators' free-text notes and
# the review texts are read but never printed.
import re
rv = pd.read_csv(REL / f"drugscom_plus_ai_labels_{PROTOCOL_ID}.csv", usecols=["uniqueID", "review"]).set_index("uniqueID").reindex(H.uniqueID).review.astype(str).str.lower().values
lab = {"LLM": H.LLM, "A1": H.A1, "A2": H.A2}; asp = {(k, a): H[c] for a in ASPECTS for k, c in (("LLM", f"LLM_{a}"), ("A1", f"A1_{a}"), ("A2", f"A2_{a}"))}
def agree(mask, tag):
    out = []
    for x, y in (("A1", "A2"), ("LLM", "A1"), ("LLM", "A2")):
        u, v = lab[x][mask], lab[y][mask]; hi = [CLASSES.index(s) for s in u]; hj = [CLASSES.index(s) for s in v]
        r = {"subset": tag, "n": int(mask.sum()), "pair": f"{x}-{y}", "holistic_pct": round(100 * np.mean(np.array(hi) == np.array(hj)), 1),
             "holistic_qwk": round(cohen_kappa_score(hi, hj, labels=range(5), weights="quadratic"), 3) if len(set(hi) | set(hj)) > 1 else np.nan}
        for a in ASPECTS:
            u, v = asp[(x, a)][mask], asp[(y, a)][mask]
            r[f"{a}_kappa"] = round(cohen_kappa_score(u, v, labels=A3), 3) if len(set(u) | set(v)) > 1 else np.nan
        out.append(r)
    return out
rows = agree(np.ones(len(H), bool), "all reviews")
for n in (1, 2):
    f = (H[f"A{n}_uncertain"] == "YES").values; rows += agree(f, f"annotator {n} flagged uncertain") + agree(~f, f"annotator {n} not flagged")
CUES = {"withdrawal": r"withdraw|taper|wean|coming off|come off|came off|get(ting)? off|quit(ting)?|discontinu|stopp?(ed|ing)? (taking|using|the|it|this)",
        "temporal change": r"at first|no longer|stopped working|wore off|used to|after a (week|month|year)|initially|over time|eventually|came back|the first (few|couple|week|month|day)"}
for name, rx in CUES.items():
    c = pd.Series(rv).str.contains(rx, regex=True).values; rows += agree(c, f"{name} cue in review") + agree(~c, f"no {name} cue")
s8 = pd.DataFrame(rows); print(s8.to_string(index=False)); save(s8, "agreement_within_subsets.csv", index=False)
c = pd.Series(rv).str.contains(CUES["withdrawal"], regex=True).values
for a in ("burden", "safety"):
    for k in ("A1", "A2"):
        print(f"\nwithdrawal cue ({int(c.sum())} reviews), {a}: annotation (rows) against {k} (columns)\n" + pd.crosstab(asp[("LLM", a)][c].rename("annotation"), asp[(k, a)][c].rename(k)).to_string())

/tmp/ipykernel_2012/4106573091.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  c = pd.Series(rv).str.contains(rx, regex=True).values; rows += agree(c, f"{name} cue in review") + agree(~c, f"no {name} cue")
/tmp/ipykernel_2012/4106573091.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  c = pd.Series(rv).str.contains(rx, regex=True).values; rows += agree(c, f"{name} cue in review") + agree(~c, f"no {name} cue")


                       subset   n   pair  holistic_pct  holistic_qwk  efficacy_kappa  safety_kappa  burden_kappa  cost_kappa
                  all reviews 500  A1-A2          44.4         0.770           0.509         0.619         0.368       0.704
                  all reviews 500 LLM-A1          43.4         0.792           0.379         0.587         0.251       0.720
                  all reviews 500 LLM-A2          75.0         0.887           0.495         0.606         0.398       0.806
annotator 1 flagged uncertain  20  A1-A2          35.0         0.592           0.269         0.479         0.214       0.669
annotator 1 flagged uncertain  20 LLM-A1          30.0         0.482           0.160         0.643         0.231       0.853
annotator 1 flagged uncertain  20 LLM-A2          75.0         0.703           0.455         0.468         0.464       0.836
      annotator 1 not flagged 480  A1-A2          44.8         0.776           0.518         0.623         0.374       0.705


/tmp/ipykernel_2012/4106573091.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  c = pd.Series(rv).str.contains(CUES["withdrawal"], regex=True).values
